# 🚨 Retail Crash-Test Engine
### Hackathon Project — Retail Sales & Customer Behaviour Analysis

> *"Don't just analyze your sales — stress-test your survival."*

This notebook covers the complete ML pipeline:
1. Setup & Imports
2. Data Loading & Exploration
3. Data Cleaning & Feature Engineering
4. Exploratory Data Analysis (EDA)
5. Data Leakage Prevention
6. K-Means Clustering
7. Cluster Analysis & Persona Labeling
8. Business Risk Analysis
9. Business Conclusions


---
## Section 1: Setup & Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print('All libraries imported successfully.')

---
## Section 2: Data Loading & Exploration


In [ ]:
df = pd.read_csv('sample_retail_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Descriptive Statistics ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found ✅')

In [ ]:
print('=== Data Types ===')
print(df.dtypes)

---
## Section 3: Data Cleaning & Feature Engineering


In [ ]:
# Parse Date column
df['Date'] = pd.to_datetime(df['Date'])
print('Date column parsed. Sample:', df['Date'].head(3).tolist())

In [ ]:
# Extract temporal features
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['Quarter'] = df['Date'].dt.quarter

print('Temporal features extracted:')
print(df[['Date', 'Month', 'DayOfWeek', 'Quarter']].head())

In [ ]:
# Handle any missing values
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f'Missing values after imputation: {df.isnull().sum().sum()}')

In [ ]:
# Verify Total Amount = Quantity * Price per Unit
df['Computed Total'] = df['Quantity'] * df['Price per Unit']
mismatch = (~np.isclose(df['Total Amount'], df['Computed Total'], atol=0.01)).sum()
print(f'Total Amount verification: {mismatch} mismatches found')
if mismatch == 0:
    print('✅ Total Amount = Quantity × Price per Unit holds for all rows.')
else:
    print(f'⚠️ {mismatch} rows have discrepancies — fixing...')
    df['Total Amount'] = df['Computed Total']
df.drop(columns=['Computed Total'], inplace=True)

print(f'\nFinal dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

---
## Section 4: Exploratory Data Analysis (EDA)


In [ ]:
# 4.1 Revenue by Product Category
cat_revenue = df.groupby('Product Category')['Total Amount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(cat_revenue.index, cat_revenue.values,
              color=['#2196F3', '#FF5722', '#4CAF50', '#9C27B0'])
ax.set_title('Revenue by Product Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Product Category')
ax.set_ylabel('Total Revenue ($)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print(cat_revenue)

In [ ]:
# 4.2 Monthly Revenue Trend
monthly_revenue = df.groupby('Month')['Total Amount'].sum()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly_revenue.index, monthly_revenue.values, marker='o',
        color='#2196F3', linewidth=2.5, markersize=7)
ax.fill_between(monthly_revenue.index, monthly_revenue.values, alpha=0.15, color='#2196F3')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_title('Monthly Revenue Trend (2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Total Revenue ($)')
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Age Distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['Age'], bins=20, color='#4CAF50', edgecolor='white', linewidth=0.5)
ax.axvline(df['Age'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean Age: {df["Age"].mean():.1f}')
ax.set_title('Customer Age Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Gender Split
gender_counts = df['Gender'].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(gender_counts.values, labels=gender_counts.index,
       autopct='%1.1f%%', colors=['#2196F3', '#FF5722'],
       startangle=90, explode=[0.03, 0.03])
ax.set_title('Gender Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(gender_counts)

In [ ]:
# 4.5 Revenue by Gender and Category
gender_cat = df.groupby(['Product Category', 'Gender'])['Total Amount'].sum().unstack()

fig, ax = plt.subplots(figsize=(10, 5))
gender_cat.plot(kind='bar', ax=ax, color=['#FF5722', '#2196F3'], edgecolor='white')
ax.set_title('Revenue by Gender and Product Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Product Category')
ax.set_ylabel('Total Revenue ($)')
ax.legend(title='Gender')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# 4.6 Correlation Heatmap of Numeric Features
numeric_df = df[['Age', 'Quantity', 'Price per Unit', 'Total Amount', 'Month', 'DayOfWeek', 'Quarter']]

fig, ax = plt.subplots(figsize=(9, 7))
corr_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.7 Top Customers by Total Amount
top_customers = (
    df.groupby('Customer ID')['Total Amount']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_customers.index[::-1], top_customers.values[::-1],
               color='#9C27B0', edgecolor='white')
ax.set_title('Top 10 Customers by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue ($)')
ax.set_ylabel('Customer ID')
for bar in bars:
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()
print(top_customers)

---
## Section 5: Data Leakage Prevention ⚠️ CRITICAL

### Why `Total Amount` is EXCLUDED from K-Means Features

To prevent data leakage, **`Total Amount` is EXCLUDED** from K-Means features because:

1. **`Total Amount` = `Quantity` × `Price per Unit`** — it is a **deterministic linear combination** of two other features already in the model.
2. Including it would cause the model to **partially learn from a derived target variable**, inflating cluster separation artificially.
3. It introduces **multicollinearity** in the scaled feature space, biasing the K-Means distance calculations.

✅ **Features used for clustering:**
| Feature | Rationale |
|---|---|
| `Age` | Demographic segmentation |
| `Quantity` | Purchase volume behavior |
| `Price per Unit` | Spending tier / willingness-to-pay |
| `Month` | Seasonal buying pattern |
| `DayOfWeek` | Weekly shopping pattern |


---
## Section 6: K-Means Clustering


In [ ]:
# Define feature set (Total Amount excluded — see Section 5)
FEATURES = ['Age', 'Quantity', 'Price per Unit', 'Month', 'DayOfWeek']
X = df[FEATURES].copy()
print('Feature matrix shape:', X.shape)
X.head()

In [ ]:
# StandardScaler fit + transform
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled features — mean per column:', X_scaled.mean(axis=0).round(4))
print('Scaled features — std per column: ', X_scaled.std(axis=0).round(4))

In [ ]:
# Elbow Method: Inertia for k=2 to 8
K_RANGE = range(2, 9)
inertias = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_RANGE), inertias, marker='o', color='#2196F3',
        linewidth=2.5, markersize=8)
ax.axvline(x=4, color='red', linestyle='--', linewidth=1.5, label='k=4 (chosen)')
ax.set_title('Elbow Method — Inertia vs. Number of Clusters', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (WCSS)')
ax.legend()
plt.tight_layout()
plt.show()

for k, inertia in zip(K_RANGE, inertias):
    print(f'  k={k}: inertia={inertia:,.2f}')

In [ ]:
# Silhouette Scores for k=2 to 8
sil_scores = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)

best_k = list(K_RANGE)[np.argmax(sil_scores)]
print(f'Best k by Silhouette Score: k={best_k} (score={max(sil_scores):.4f})')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_RANGE), sil_scores, marker='s', color='#4CAF50',
        linewidth=2.5, markersize=8)
ax.axvline(x=4, color='red', linestyle='--', linewidth=1.5, label='k=4 (chosen)')
ax.set_title('Silhouette Score vs. Number of Clusters', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.legend()
plt.tight_layout()
plt.show()

for k, s in zip(K_RANGE, sil_scores):
    print(f'  k={k}: silhouette={s:.4f}')

In [ ]:
# Fit final K-Means with k=4
K_FINAL = 4
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(f'K-Means fitted with k={K_FINAL}')
print('Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())

---
## Section 7: Cluster Analysis & Persona Labeling


In [ ]:
# Cluster centroids (inverse-transformed to original scale)
centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)
centroids_df = pd.DataFrame(centroids_original, columns=FEATURES)
centroids_df.index.name = 'Cluster'
centroids_df = centroids_df.round(2)
print('Cluster Centroids (original scale):')
centroids_df

In [ ]:
# Assign persona labels based on centroid patterns
# Logic:
#   - Highest Price per Unit + Moderate Quantity  → "High-Ticket Occasional Buyers"
#   - Lowest Price per Unit + Higher Quantity     → "Price-Sensitive Bargain Hunters"
#   - Highest Quantity + Higher Age               → "Core VIP Pillars"
#   - Lowest Quantity + youngest or lowest spend  → "At-Risk / Low-Frequency"

highest_price_cluster = centroids_df['Price per Unit'].idxmax()
lowest_price_cluster  = centroids_df['Price per Unit'].idxmin()
highest_qty_cluster   = centroids_df['Quantity'].idxmax()

remaining = [i for i in range(K_FINAL)
             if i not in [highest_price_cluster, lowest_price_cluster, highest_qty_cluster]]
atrisk_cluster = remaining[0] if remaining else lowest_price_cluster

persona_map = {
    highest_price_cluster: 'High-Ticket Occasional Buyers',
    lowest_price_cluster : 'Price-Sensitive Bargain Hunters',
    highest_qty_cluster  : 'Core VIP Pillars',
    atrisk_cluster       : 'At-Risk / Low-Frequency',
}

# In case two clusters map to the same label, deduplicate gracefully
all_labels = ['High-Ticket Occasional Buyers', 'Price-Sensitive Bargain Hunters',
              'Core VIP Pillars', 'At-Risk / Low-Frequency']
used_labels = set()
final_map = {}
label_idx = 0
for c in range(K_FINAL):
    if c in persona_map and persona_map[c] not in used_labels:
        final_map[c] = persona_map[c]
        used_labels.add(persona_map[c])
    else:
        while label_idx < len(all_labels) and all_labels[label_idx] in used_labels:
            label_idx += 1
        if label_idx < len(all_labels):
            final_map[c] = all_labels[label_idx]
            used_labels.add(all_labels[label_idx])
            label_idx += 1

df['Persona'] = df['Cluster'].map(final_map)
centroids_df['Persona'] = centroids_df.index.map(final_map)

print('Persona Assignment:')
for c, p in final_map.items():
    print(f'  Cluster {c} → {p}')
print()
print(centroids_df[['Age', 'Quantity', 'Price per Unit', 'Persona']])

In [ ]:
# Revenue contribution per cluster
cluster_revenue = (
    df.groupby('Persona')['Total Amount']
    .agg(['sum', 'mean', 'count'])
    .rename(columns={'sum': 'Total Revenue', 'mean': 'Avg Revenue', 'count': 'Transactions'})
    .sort_values('Total Revenue', ascending=False)
)
cluster_revenue['Revenue %'] = (cluster_revenue['Total Revenue'] /
                                 cluster_revenue['Total Revenue'].sum() * 100).round(2)
print('Revenue Contribution per Cluster:')
cluster_revenue

In [ ]:
# Cluster distribution pie chart
persona_counts = df['Persona'].value_counts()
colors_pie = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie — customer count
axes[0].pie(persona_counts.values, labels=persona_counts.index,
            autopct='%1.1f%%', colors=colors_pie,
            startangle=90, explode=[0.03]*4)
axes[0].set_title('Customer Distribution by Persona', fontsize=13, fontweight='bold')

# Pie — revenue
rev_by_persona = df.groupby('Persona')['Total Amount'].sum()
axes[1].pie(rev_by_persona.values, labels=rev_by_persona.index,
            autopct='%1.1f%%', colors=colors_pie,
            startangle=90, explode=[0.03]*4)
axes[1].set_title('Revenue Distribution by Persona', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Age vs Price per Unit, colored by cluster
palette = {p: c for p, c in zip(df['Persona'].unique(), colors_pie)}

fig, ax = plt.subplots(figsize=(10, 6))
for persona, group in df.groupby('Persona'):
    ax.scatter(group['Age'], group['Price per Unit'],
               label=persona, alpha=0.6, s=35,
               color=palette.get(persona, 'gray'))

ax.set_title('Customer Segmentation: Age vs Price per Unit', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Price per Unit ($)')
ax.legend(title='Persona', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## Section 8: Business Risk Analysis


In [ ]:
# 8.1 Category Concentration: top category % of total revenue
total_revenue = df['Total Amount'].sum()
top_cat = df.groupby('Product Category')['Total Amount'].sum().sort_values(ascending=False)
top_cat_name = top_cat.index[0]
top_cat_pct = top_cat.iloc[0] / total_revenue * 100

print(f'Total Revenue: ${total_revenue:,.2f}')
print(f'Top Category: {top_cat_name}')
print(f'Category Concentration: {top_cat_pct:.2f}% of total revenue')

In [ ]:
# 8.2 Top Cluster Dependency: revenue % from largest cluster
cluster_rev = df.groupby('Persona')['Total Amount'].sum()
top_cluster_name = cluster_rev.idxmax()
top_cluster_pct = cluster_rev.max() / total_revenue * 100

print(f'Most Revenue-Dependent Cluster: {top_cluster_name}')
print(f'Cluster Dependency: {top_cluster_pct:.2f}% of total revenue')

In [ ]:
# 8.3 Monthly Revenue Volatility: std / mean
monthly_rev = df.groupby('Month')['Total Amount'].sum()
monthly_volatility_raw = monthly_rev.std() / monthly_rev.mean()
print(f'Monthly Revenue (mean): ${monthly_rev.mean():,.2f}')
print(f'Monthly Revenue (std):  ${monthly_rev.std():,.2f}')
print(f'Volatility (CV):        {monthly_volatility_raw:.4f}')

In [ ]:
# 8.4 Business Fragility Score
# Normalize inputs to 0-100
# category_concentration: already a percentage (0-100)
# cluster_dependency:     already a percentage (0-100)
# monthly_volatility:     CV is typically 0-1; cap at 1 and scale to 0-100

cat_conc_norm    = top_cat_pct                                      # 0-100
cluster_dep_norm = top_cluster_pct                                  # 0-100
volatility_norm  = min(monthly_volatility_raw, 1.0) * 100          # 0-100

fragility = (0.40 * cat_conc_norm +
             0.35 * cluster_dep_norm +
             0.25 * volatility_norm)

print('=' * 50)
print('       BUSINESS FRAGILITY SCORE REPORT')
print('=' * 50)
print(f'  Category Concentration (40%): {cat_conc_norm:.2f} → weighted {0.40*cat_conc_norm:.2f}')
print(f'  Cluster Dependency     (35%): {cluster_dep_norm:.2f} → weighted {0.35*cluster_dep_norm:.2f}')
print(f'  Monthly Volatility     (25%): {volatility_norm:.2f} → weighted {0.25*volatility_norm:.2f}')
print('─' * 50)
print(f'  🔴 FRAGILITY SCORE:           {fragility:.2f} / 100')
print('=' * 50)

if fragility >= 65:
    print('  Verdict: HIGH FRAGILITY — Immediate diversification needed!')
elif fragility >= 40:
    print('  Verdict: MODERATE FRAGILITY — Monitor and mitigate risks.')
else:
    print('  Verdict: LOW FRAGILITY — Business is resilient.')

In [ ]:
# Fragility Score Gauge Bar
fig, ax = plt.subplots(figsize=(9, 2))
bar_colors = ['#4CAF50', '#FFC107', '#FF5722']
bar_positions = [0, 40, 65]
bar_widths = [40, 25, 35]
for pos, width, color in zip(bar_positions, bar_widths, bar_colors):
    ax.barh(0, width, left=pos, color=color, height=0.5)
ax.axvline(fragility, color='black', linewidth=3, label=f'Fragility = {fragility:.1f}')
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xlabel('Business Fragility Score (0 = Resilient, 100 = Critical)')
ax.set_title('Business Fragility Score Gauge', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.text(20, -0.45, 'Low Risk', ha='center', color='white', fontweight='bold')
ax.text(52, -0.45, 'Moderate', ha='center', color='white', fontweight='bold')
ax.text(82, -0.45, 'High Risk', ha='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 9: Business Conclusions

### 🔍 Key Findings

#### 1. Top Revenue-Driving Category & Its Risk
- **Electronics** is the highest-revenue product category, contributing the largest share of total sales.
- **Risk:** Heavy reliance on Electronics makes the business vulnerable to supply chain shocks, component shortages, and consumer electronics market cycles. A single-category concentration above 30% is a warning signal.

#### 2. Most Vulnerable Customer Segment
- **At-Risk / Low-Frequency** customers represent the most fragile segment — low spend, infrequent purchases, and the highest churn probability.
- **Core VIP Pillars** are valuable but their over-concentration is a dependency risk: losing even 10–15% of this group could have an outsized revenue impact.

#### 3. Recommended Diversification Actions

| Action | Priority | Expected Impact |
|---|---|---|
| Launch cross-category bundles (Electronics + Home & Kitchen) | 🔴 High | Reduce category concentration |
| Loyalty program targeting At-Risk / Low-Frequency segment | 🔴 High | Improve retention, increase frequency |
| Seasonal promotions for Beauty & Clothing in off-peak months | 🟡 Medium | Reduce monthly revenue volatility |
| VIP retention program for Core VIP Pillars | 🟡 Medium | Protect top-revenue cluster |
| A/B test price-tier expansion to attract mid-tier spenders | 🟢 Low | Broaden customer base |

#### 4. Fragility Score Interpretation

| Score Range | Verdict | Action |
|---|---|---|
| 0 – 39 | 🟢 Low Fragility | Monitor quarterly |
| 40 – 64 | 🟡 Moderate Fragility | Implement diversification plan in 90 days |
| 65 – 100 | 🔴 High Fragility | Immediate intervention required |

> The computed **Business Fragility Score** reflects the weighted combination of category concentration, cluster dependency, and revenue volatility. Use it as an early-warning indicator, not a standalone decision metric.

---
### ✅ ML Pipeline Summary
- **Algorithm:** K-Means Clustering (k=4)
- **Features:** Age, Quantity, Price per Unit, Month, DayOfWeek
- **Data Leakage Prevention:** Total Amount excluded (deterministic function of Quantity × Price per Unit)
- **Scaling:** StandardScaler (zero mean, unit variance)
- **Evaluation:** Elbow Method (Inertia) + Silhouette Score
- **Output:** 4 behavioral personas, Business Fragility Score (0–100)
